[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Nearest Neighbors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)

named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)

districts = forecast_districts()
print("geopandas", gpd.__version__, "|", len(stations), "made-up stations,", len(named), "named stations,",
      len(districts), "made-up districts")

geopandas 1.2.0 | 300 made-up stations, 5 named stations, 8 made-up districts


The worked examples' tables in meters, rebuilt here so that each answer runs on its own. The stations and the districts are made up.

In [2]:
stations_m = stations.to_crs(25833)
named_m = named.to_crs(25833)

**1.** Find the station farthest from its nearest named station, and print its id, the named station and the distance in kilometers.

In [3]:
nearest = gpd.sjoin_nearest(stations_m, named_m[["name", "geometry"]], distance_col="distance_m")
far = nearest.assign(km=(nearest["distance_m"] / 1000).round(1)).nlargest(1, "km")
print(far[["station", "name", "km"]].to_string(index=False))

station   name    km
  NS233 Tromso 577.2


`nlargest` keeps the row with the largest distance, the station worst served by the five named stations.

**2.** Give each named station its nearest country other than Norway, and print the country and the distance in kilometers.

In [4]:
countries_m = countries.to_crs(25833)
abroad = countries_m[countries_m["ADMIN"] != "Norway"][["ADMIN", "geometry"]]
neighbor = gpd.sjoin_nearest(named_m, abroad, distance_col="distance_m")
neighbor.assign(km=(neighbor["distance_m"] / 1000).round(1))[["name", "ADMIN", "km"]]

,name,ADMIN,km
0,Bergen,Sweden,361.7
1,Oslo,Sweden,63.2
2,Svalbard,Finland,978.0
3,Tromso,Sweden,83.7
4,Kirkenes,Russia,11.7


Leaving Norway out of the right table makes the join look past it to the next closest country. The distance is measured to that country's edge, so Kirkenes is 11.7 km from Russia.

**3.** Count how many stations have a named station within 200 km, with a left join and `max_distance`, and print how many have none.

In [5]:
limited = gpd.sjoin_nearest(stations_m, named_m[["name", "geometry"]], how="left", max_distance=200_000)
print("within 200 km:", limited["name"].notna().sum())
print("none within 200 km:", limited["name"].isna().sum())

within 200 km: 174
none within 200 km: 126


The limit is in meters, so 200 km is `200_000`. The left join keeps every station, and the ones with nothing inside the limit have `NaN` for `name`.

**4.** Pair every station in the Mid district with its nearest station outside Mid, and print the five shortest pairs with their distances in kilometers.

In [6]:
with_district = gpd.sjoin(stations_m, districts)
mid = with_district[with_district["district"] == "Mid"][["station", "geometry"]]
outside = with_district[with_district["district"] != "Mid"][["station", "district", "geometry"]]
pairs = gpd.sjoin_nearest(mid, outside, distance_col="distance_m")
pairs["km"] = (pairs["distance_m"] / 1000).round(1)
pairs.nsmallest(5, "km")[["station_left", "station_right", "district", "km"]]

,station_left,station_right,district,km
295,NS296,NS223,East,4.0
88,NS089,NS106,East,18.4
165,NS166,NS223,East,18.8
24,NS025,NS090,West,19.1
133,NS134,NS106,East,21.7


`sjoin` gives each station its district first. The two `station` columns get the suffixes `_left` and `_right`, and `district` comes from the stations outside Mid.

**5.** For the `airport` stations only, find each one's nearest `airport` station, and print the mean of those distances in kilometers.

In [7]:
airports = stations_m[stations_m["owner"] == "airport"][["station", "geometry"]]
pairs = gpd.sjoin_nearest(airports, airports, distance_col="distance_m", exclusive=True)
print(len(airports), "airport stations")
print("mean distance to the nearest other one:", round(pairs["distance_m"].mean() / 1000, 1), "km")

33 airport stations
mean distance to the nearest other one: 64.2 km


`exclusive=True` stops each station matching itself. The 33 airport stations are about a tenth of the register, and their mean distance to the nearest other one, 64.2 km, is more than three times the whole network's mean of 17.7 km.

---

&#8592; **Back to:** [Nearest Neighbors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/09-nearest-neighbors.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
